# 20 · PPO: the clipped objective, value loss, entropy, and KL estimators

**Needed for:** PPO (RL track, step R2) and PPO-based RLHF (step R3). The KL estimators are used by GRPO too
(notebook 22).

Policy gradient uses each batch of experience once. PPO reuses it for several epochs of updates, which is only
safe if the policy doesn't move too far from the one that collected the data. The **probability ratio**
$\rho_t = \pi_\theta(a_t\mid s_t) / \pi_{old}(a_t\mid s_t)$ measures how far it has moved, and the loss is clipped
so there's no incentive to push $\rho$ outside $[1-\epsilon, 1+\epsilon]$:

$$\mathcal{L}^{CLIP} = -\mathbb{E}\Big[\min\big(\rho_t A_t,\ \text{clip}(\rho_t, 1-\epsilon, 1+\epsilon)\,A_t\big)\Big]$$

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. The clipped policy loss, and where its gradient is zero

The `min` makes the objective pessimistic:
- $A > 0$: the gain is capped once $\rho > 1+\epsilon$ (gradient becomes 0), but the loss is *not* capped below.
- $A < 0$: the gain is capped once $\rho < 1-\epsilon$, not above.

Work with log-probabilities: `ratio = exp(logp - old_logp)`. It's numerically safer than dividing probabilities.

In [ ]:
def ppo_policy_loss(logp, old_logp, adv, mask, clip_eps=0.2):
    """All (B, T). Returns the scalar loss averaged over masked steps, and the fraction of clipped steps."""
    ratio = torch.exp(logp - old_logp)
    unclipped = ratio * adv
    clipped = ratio.clamp(1 - clip_eps, 1 + clip_eps) * adv
    loss = -torch.min(unclipped, clipped)
    clip_frac = ((ratio - 1).abs() > clip_eps).float()
    return (loss * mask).sum() / mask.sum(), (clip_frac * mask).sum() / mask.sum()


def test_ppo_loss(fn):
    ratios = torch.tensor([[0.5, 1.0, 1.5, 0.5, 1.0, 1.5]])
    adv = torch.tensor([[1.0, 1.0, 1.0, -1.0, -1.0, -1.0]])
    logp = ratios.log().requires_grad_(True)
    loss, clip_frac = fn(logp, torch.zeros_like(logp), adv, torch.ones_like(adv))
    loss.backward()
    grad_zero = (logp.grad.abs() < 1e-9).flatten().tolist()
    # A>0: clipped (no gradient) only when ratio > 1.2; A<0: only when ratio < 0.8
    assert grad_zero == [False, False, True, True, False, False], grad_zero
    torch.testing.assert_close(clip_frac, torch.tensor(4 / 6))


test_ppo_loss(ppo_policy_loss)
print("gradient is zero exactly where the clip is active in the pessimistic direction")

## 2. KL estimators (Schulman's k1, k2, k3)

To penalize drift from a reference policy you need $\text{KL}(\pi \,\|\, \pi_{ref})$ from samples $x\sim\pi$ only.
With $r = \pi_{ref}(x) / \pi(x)$:

| Estimator | Formula | Unbiased? | Always ≥ 0? |
|---|---|---|---|
| k1 | $-\log r$ | yes | no (high variance) |
| k2 | $\tfrac12(\log r)^2$ | no (low bias when close) | yes |
| k3 | $(r - 1) - \log r$ | yes | yes, lower variance |

GRPO uses k3 as a per-token penalty. Check with two Gaussians, where the true KL is known in closed form.

In [ ]:
def kl_estimators(logp, logp_ref):
    """Per-sample KL estimates for samples drawn from p. Returns (k1, k2, k3)."""
    log_r = logp_ref - logp
    return -log_r, 0.5 * log_r**2, torch.expm1(log_r) - log_r


def test_kl(fn):
    torch.manual_seed(0)
    p, q = torch.distributions.Normal(0.0, 1.0), torch.distributions.Normal(0.3, 1.0)
    x = p.sample((400_000,))
    k1, k2, k3 = fn(p.log_prob(x), q.log_prob(x))
    true_kl = torch.distributions.kl_divergence(p, q)
    torch.testing.assert_close(k1.mean(), true_kl, atol=5e-3, rtol=0)
    torch.testing.assert_close(k3.mean(), true_kl, atol=5e-3, rtol=0)
    assert (k3 >= 0).all() and (k2 >= 0).all() and (k1 < 0).any()


test_kl(kl_estimators)
p = torch.distributions.Normal(0.0, 1.0)
for mu in (0.1, 1.0):
    q = torch.distributions.Normal(mu, 1.0)
    x = p.sample((200_000,))
    ks = kl_estimators(p.log_prob(x), q.log_prob(x))
    print(f"true KL {torch.distributions.kl_divergence(p, q).item():.4f}   " +
          "   ".join(f"{n}: mean {k.mean().item():.4f} std {k.std().item():.4f}" for n, k in zip(("k1", "k2", "k3"), ks)))

## 3. Full PPO on the corridor from notebook 19

Actor and critic share nothing here, for clarity. Each iteration: collect a batch with the current policy, compute
GAE advantages with the critic, then run several epochs of minibatch updates on the clipped loss + value loss −
entropy bonus. Advantages are normalized per batch.

In [ ]:
N_POS, MAX_STEPS = 10, 30


def rollout(policy, B):
    s = torch.zeros(B, dtype=torch.long)
    alive = torch.ones(B, dtype=torch.bool)
    buf = {k: [] for k in ("obs", "act", "logp", "rew", "done", "alive")}
    with torch.no_grad():
        for _ in range(MAX_STEPS):
            obs = F.one_hot(s, N_POS).float()
            dist = torch.distributions.Categorical(logits=policy(obs))
            a = dist.sample()
            nxt = (s + 2 * a - 1).clamp(0, N_POS - 1)
            reached = nxt == N_POS - 1
            buf["obs"].append(obs)
            buf["act"].append(a)
            buf["logp"].append(dist.log_prob(a))
            buf["rew"].append(torch.where(reached, 1.0, -0.01) * alive)
            buf["done"].append((reached & alive).float())
            buf["alive"].append(alive.float())
            s = torch.where(alive, nxt, s)
            alive = alive & ~reached
    return {k: torch.stack(v, dim=1) for k, v in buf.items()}


def gae(rewards, values, dones, gamma, lam):
    adv = torch.zeros_like(rewards)
    running = torch.zeros_like(rewards[..., 0])
    for t in reversed(range(rewards.shape[-1])):
        not_done = 1 - dones[..., t]
        delta = rewards[..., t] + gamma * values[..., t + 1] * not_done - values[..., t]
        running = delta + gamma * lam * not_done * running
        adv[..., t] = running
    return adv


torch.manual_seed(0)
actor = nn.Sequential(nn.Linear(N_POS, 64), nn.Tanh(), nn.Linear(64, 2))
critic = nn.Sequential(nn.Linear(N_POS, 64), nn.Tanh(), nn.Linear(64, 1))
opt = torch.optim.Adam([*actor.parameters(), *critic.parameters()], lr=3e-3)
for it in range(31):
    b = rollout(actor, 64)
    with torch.no_grad():
        values = critic(b["obs"]).squeeze(-1)
        values = torch.cat([values, torch.zeros(64, 1)], dim=1) * torch.cat([b["alive"], torch.zeros(64, 1)], dim=1)
        adv = gae(b["rew"], values, b["done"], 0.99, 0.95)
        returns = adv + values[:, :-1]
    m = b["alive"].bool()
    obs, act, old_logp, adv_f, ret_f = b["obs"][m], b["act"][m], b["logp"][m], adv[m], returns[m]
    adv_f = (adv_f - adv_f.mean()) / (adv_f.std() + 1e-8)
    for epoch in range(4):
        for idx in torch.randperm(len(obs)).split(256):
            dist = torch.distributions.Categorical(logits=actor(obs[idx]))
            logp = dist.log_prob(act[idx])
            pl, clip_frac = ppo_policy_loss(logp, old_logp[idx], adv_f[idx], torch.ones_like(logp))
            vl = F.mse_loss(critic(obs[idx]).squeeze(-1), ret_f[idx])
            loss = pl + 0.5 * vl - 0.01 * dist.entropy().mean()
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_([*actor.parameters(), *critic.parameters()], 0.5)
            opt.step()
    if it % 5 == 0:
        k1, _, k3 = kl_estimators(old_logp, torch.distributions.Categorical(logits=actor(obs)).log_prob(act).detach())
        print(f"iter {it:2d}  mean return {b['rew'].sum(1).mean().item():6.3f}  clip frac {clip_frac.item():.2f}  "
              f"KL(old||new) k3 {k3.mean().item():.4f}")

Watch the diagnostics you'd monitor in a real run: **clip fraction** (a few percent to ~20% is healthy while learning; stuck near
0 before convergence means the learning rate could be higher; very high means updates are too aggressive. Here it
drops to 0 because the policy has converged and stops changing.) and **KL between the old and new
policy** (spikes precede collapse).

## Things to be able to explain

- Why normalize advantages per batch? (Keeps the step size stable as reward scales change.)
- In RLHF with PPO, which four models are in memory? (Policy, reference, reward model, value model.) Which ones does
  GRPO remove? (The value model; notebook 22.)
- Where does the KL penalty go in RLHF-PPO: into the reward (per token, InstructGPT) or into the loss (GRPO)?

## Drill

In [ ]:
def my_ppo_policy_loss(logp, old_logp, adv, mask, clip_eps=0.2):
    raise NotImplementedError


def my_kl_estimators(logp, logp_ref):
    raise NotImplementedError


drill(test_ppo_loss, my_ppo_policy_loss)
drill(test_kl, my_kl_estimators)